Import dataset and libraries


In [535]:
import pandas as pd
import tqdm
import numpy as np
import torch
from torch import nn
import os

In [536]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [537]:
!rm -rf /content/dataset
!unzip -q "/content/drive/MyDrive/masked_sets.zip" -d "/content/dataset"

In [538]:
from pathlib import Path
data_root = Path("/content/dataset/masked_sets")

Prepare data



In [539]:
from sklearn.model_selection import train_test_split

def load_flat_set(root, class_to_idx):
    X_rgb, X_depth, y = [], [], []
    rgb_paths = sorted([p for p in root.glob("*_rgb*.png") if p.is_file()])

    for rgb_path in rgb_paths:
        depth_path = Path(str(rgb_path).replace("_rgb", "_depth"))

        if depth_path.exists():
            full_name = rgb_path.name

            if "_random_rgb_" in full_name:
                label = full_name.split("_random_rgb_")[0].replace("_masked", "").replace("_clean", "")
            elif "_scene_" in full_name:
                label = full_name.split("_scene_")[0]
            else:
                label = full_name.split("_")[0]

            if label in class_to_idx:
                X_rgb.append(str(rgb_path))
                X_depth.append(str(depth_path))
                y.append(class_to_idx[label])

    return X_rgb, X_depth, y

# Path for training set
train_root = data_root / "train"

# Build class list from filenames in the flat train folder
classes = sorted({
    p.name.split("_random_rgb_")[0].replace("_masked", "").replace("_clean", "")
    for p in train_root.glob("*_rgb_*.png")
    if p.is_file()
})
#target_classes = ['bowl', 'cap', 'cereal_box', 'coffee_mug', 'soda_can']
#class_to_idx = {c: i for i, c in enumerate(target_classes)}
class_to_idx = {c: i for i, c in enumerate(classes)}

# Load training set (paired)
X_rgb_all, X_depth_all, y_all = load_flat_set(train_root, class_to_idx)
print("Loaded training samples:", len(y_all))

# Split training set into train/val
X_rgb_train, X_rgb_val, X_depth_train, X_depth_val, y_train, y_val = train_test_split(
    X_rgb_all, X_depth_all, y_all,
    test_size=0.2,
    random_state=1234,
    stratify=y_all
)
print("Train samples:", len(y_train), "Val samples:", len(y_val))

# Paths for test sets
root_low  = data_root / "test" / "low_occlusion"
root_med  = data_root / "test" / "medium_occlusion"
root_high = data_root / "test" / "high_occlusion"

# Load test sets using the same classes & mapping
X_rgb_test_low,  X_depth_test_low,  y_test_low  = load_flat_set(root_low, class_to_idx)
X_rgb_test_med,  X_depth_test_med,  y_test_med  = load_flat_set(root_med, class_to_idx)
X_rgb_test_high, X_depth_test_high, y_test_high = load_flat_set(root_high, class_to_idx)

# Print test set sizes
print("Low occlusion test samples:", len(y_test_low))
print("Med occlusion test samples:", len(y_test_med))
print("High occlusion test samples:", len(y_test_high))

Loaded training samples: 10000
Train samples: 8000 Val samples: 2000
Low occlusion test samples: 300
Med occlusion test samples: 300
High occlusion test samples: 300


Create Fusion Model


In [540]:
import torch
import torch.nn as nn
import torchvision.models as models

class RGBD_Fusion_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # RGB branch
        self.rgb_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        rgb_feature_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()

        # Depth branch
        self.depth_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        depth_feature_dim = self.depth_backbone.fc.in_features
        self.depth_backbone.fc = nn.Identity()

        # if depth is 1-channel, change first conv layer
        old_weight = self.depth_backbone.conv1.weight.data
        self.depth_backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.depth_backbone.conv1.weight.data = old_weight.mean(dim=1, keepdim=True)

        # fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(rgb_feature_dim + depth_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)

        fused_feat = torch.cat([rgb_feat, depth_feat], dim=1)
        out = self.classifier(fused_feat)
        return out

Train and test CNN models and fuse features from both

In [541]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for rgb_imgs, depth_imgs, labels in tqdm.tqdm(dataloader):
        rgb_imgs = rgb_imgs.to(device)
        depth_imgs = depth_imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(rgb_imgs, depth_imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Total Loss = {total_loss / len(dataloader):.4f}")

def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for rgb_imgs, depth_imgs, labels in dataloader:
            rgb_imgs = rgb_imgs.to(device)
            depth_imgs = depth_imgs.to(device)
            labels = labels.to(device)

            outputs = model(rgb_imgs, depth_imgs)
            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100.0 * correct / total if total > 0 else 0.0
    return acc

In [542]:
# Instantiate separate models for RGB and depth
fusion_model = RGBD_Fusion_Model(num_classes=len(classes)).to(device)
for param in fusion_model.depth_backbone.parameters():
    param.requires_grad = False


In [543]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import torch
import copy

print(device)

# fixed image size for batching
IMG_SIZE = (224, 224)

train_aug = A.Compose([
    A.LongestMaxSize(max_size=224),
    A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, fill=0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.Downscale(scale_range=(0.15, 0.25), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'depth':'mask'})

eval_aug = A.Compose([
    A.LongestMaxSize(max_size=224),
    A.PadIfNeeded(min_height=224, min_width=224, border_mode=cv2.BORDER_CONSTANT, fill=0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'depth':'mask'})

class RGBDepthDataset(Dataset):
    def __init__(self, rgb_paths, depth_paths, labels, aug=None):
        self.rgb_paths = rgb_paths
        self.depth_paths = depth_paths
        self.labels = labels
        self.aug = aug

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        rgb_path = self.rgb_paths[idx]
        depth_path = self.depth_paths[idx]

        # Load RGB
        rgb_image = np.array(Image.open(rgb_path).convert("RGB"))

        # Load depth
        depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        depth_image = depth_image.astype(np.float32)
        mask = depth_image > 0
        if np.any(mask):
            obj_depths = depth_image[mask]
            median_d = np.median(obj_depths)
            depth_image[mask] = depth_image[mask] - median_d + 1000.0

        # clip, normalize
        depth_image = np.clip(depth_image, 500, 2000)
        depth_image = (depth_image - 500) / 1500.0

        # Apply Albumentations
        if self.aug:
            augmented = self.aug(image=rgb_image, depth=depth_image)
            rgb_image = augmented["image"]
            depth_image = augmented["depth"]
        else:
            rgb_image = ToTensorV2()(image=rgb_image)["image"]
            depth_image = torch.tensor(depth_image, dtype=torch.float32)

        # Ensure depth tensor shape is [1,H,W]
        if isinstance(depth_image, np.ndarray):
            depth_image = torch.tensor(depth_image, dtype=torch.float32)

        if depth_image.ndim == 2:
            depth_image = depth_image.unsqueeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return rgb_image, depth_image, label

# Create Datasets
train_dataset = RGBDepthDataset(X_rgb_train, X_depth_train, y_train, aug=train_aug)
val_dataset = RGBDepthDataset(X_rgb_val, X_depth_val, y_val, aug=eval_aug)
test_dataset_low = RGBDepthDataset(X_rgb_test_low, X_depth_test_low, y_test_low, aug=eval_aug)
test_dataset_med = RGBDepthDataset(X_rgb_test_med, X_depth_test_med, y_test_med, aug=eval_aug)
test_dataset_high = RGBDepthDataset(X_rgb_test_high, X_depth_test_high, y_test_high, aug=eval_aug)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_low = DataLoader(test_dataset_low, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_med = DataLoader(test_dataset_med, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_high = DataLoader(test_dataset_high, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Training loop
train_losses = []
test_accuracies = []
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_fusion = torch.optim.AdamW(fusion_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fusion, mode='max', factor=0.5, patience=2)
best_val_acc = 0.0
best_model_path = "best_fusion_model.pth"
epochs = 5
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train(fusion_model, train_loader, criterion, optimizer_fusion, device)
    train_acc = evaluate(fusion_model, train_loader, device)
    print(f"Train accuracy: {train_acc:.2f}%")
    val_acc = evaluate(fusion_model, val_loader, device)
    print(f"Validation Accuracy: {val_acc:.2f}%")
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(fusion_model.state_dict(), best_model_path)
        print("Saved best model")

print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

cuda
Epoch 1/5


100%|██████████| 250/250 [00:12<00:00, 20.09it/s]

Total Loss = 2.2133


Train accuracy: 95.64%
Validation Accuracy: 95.95%
Saved best model
Epoch 2/5


100%|██████████| 250/250 [00:12<00:00, 20.46it/s]

...

100%|██████████| 250/250 [00:12<00:00, 20.82it/s]


Total Loss = 0.8146
Train accuracy: 99.80%
Validation Accuracy: 99.70%
Saved best model
Epoch 5/5


100%|██████████| 250/250 [00:12<00:00, 20.41it/s]

Total Loss = 0.7998


Train accuracy: 99.95%
Validation Accuracy: 99.75%
Saved best model
Best Validation Accuracy: 99.75%


Test the accuracy of the model in low, medium, and high occlusion

In [544]:
fusion_model.load_state_dict(torch.load("best_fusion_model.pth"))
acc_low = evaluate(fusion_model, test_loader_low, device)
acc_med = evaluate(fusion_model, test_loader_med, device)
acc_high = evaluate(fusion_model, test_loader_high, device)

print(f"Low Occlusion Accuracy: {acc_low:.2f}%")
print(f"Medium Occlusion Accuracy: {acc_med:.2f}%")
print(f"High Occlusion Accuracy: {acc_high:.2f}%")

Low Occlusion Accuracy: 82.33%
Medium Occlusion Accuracy: 86.00%
High Occlusion Accuracy: 82.00%


Test the model using real scenes

In [545]:
# the indices of the 5 target classes within the 51-class output
target_scene_classes = ['bowl', 'cap', 'cereal_box', 'coffee_mug', 'soda_can']
target_indices = [class_to_idx[cls] for cls in target_scene_classes if cls in class_to_idx]

print(f"Target Indices for Restricted Eval: {target_indices}")

def evaluate_everything(model, dataloader, device, target_indices, k=5):
    model.eval()
    correct_top1 = 0
    correct_topk = 0
    correct_restricted = 0
    total = 0

    with torch.no_grad():
        for rgb_imgs, depth_imgs, labels in dataloader:
            rgb_imgs, depth_imgs, labels = rgb_imgs.to(device), depth_imgs.to(device), labels.to(device)
            outputs = model(rgb_imgs, depth_imgs) # Shape: [Batch, 51]

            # raw accuracy
            predicted = torch.argmax(outputs, dim=1)
            correct_top1 += (predicted == labels).sum().item()

            # top-k accuracy
            _, topk_preds = outputs.topk(k, dim=1)
            correct_topk += (topk_preds == labels.view(-1, 1)).sum().item()

            # restricted to the 5 target classes accuracy
            restricted_outputs = outputs[:, target_indices]
            restricted_preds_local = torch.argmax(restricted_outputs, dim=1)
            restricted_preds_global = torch.tensor(target_indices, device=device)[restricted_preds_local]
            correct_restricted += (restricted_preds_global == labels).sum().item()

            total += labels.size(0)

    return {
        "Top-1": 100.0 * correct_top1 / total,
        "Top-5": 100.0 * correct_topk / total,
        "Restricted": 100.0 * correct_restricted / total
    }

Target Indices for Restricted Eval: [5, 8, 10, 11, 44]


In [546]:
# Paths for real world sets
real_clean = data_root / "test" / "real_world" / "Clean"
real_occluded = data_root / "test" / "real_world" / "Occluded"

# Load real world sets using the same classes & mapping
X_rgb_real_clean,  X_depth_real_clean,  y_real_clean  = load_flat_set(real_clean, class_to_idx)
X_rgb_real_occluded,  X_depth_real_occluded,  y_real_occluded  = load_flat_set(real_occluded, class_to_idx)

test_dataset_clean = RGBDepthDataset(X_rgb_real_clean,  X_depth_real_clean,  y_real_clean, aug=eval_aug)
test_dataset_occluded = RGBDepthDataset(X_rgb_real_occluded,  X_depth_real_occluded,  y_real_occluded, aug=eval_aug)


test_loader_real_clean= DataLoader(test_dataset_clean, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_real_occluded = DataLoader(test_dataset_occluded, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Print real world set sizes
print("real occlusion test samples:", len(y_real_occluded))
print("real clean test samples:", len(y_real_clean))

# evaluate real world sets
clean_results = evaluate_everything(fusion_model, test_loader_real_clean, device, target_indices)
occ_results = evaluate_everything(fusion_model, test_loader_real_occluded, device, target_indices)

print(f"{'Metric':<15} | {'Clean (%)':<10} | {'Occluded (%)':<10}")
print("-" * 45)
for metric in ["Top-1", "Top-5", "Restricted"]:
    print(f"{metric:<15} | {clean_results[metric]:<10.2f} | {occ_results[metric]:<10.2f}")

real occlusion test samples: 48
real clean test samples: 351
Metric          | Clean (%)  | Occluded (%)
---------------------------------------------
Top-1           | 30.77      | 18.75     
Top-5           | 62.39      | 37.50     
Restricted      | 73.79      | 60.42     
